<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/08_rag_faiss.ipynb)


# colab — RAG with Embeddings and FAISS

## _Retrieval-Augmented Generation on a Free GPU with Synthetic Docs_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Build a minimal retrieval-augmented generation (RAG) pipeline
  from scratch: embed documents, index with FAISS, retrieve relevant chunks,
  and generate answers with a small LLM.
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
- **Data**: Synthetic mini-documents about GPU computing, finance, and ML.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the GPU comparison.

### Roadmap

1. **Environment**: Install `sentence-transformers`, `faiss-gpu`,
   `transformers`, `torch`.
2. **Corpus**: Define 20 synthetic knowledge-base documents.
3. **Embed**: Encode documents with `all-MiniLM-L6-v2`.
4. **Index**: Build a FAISS flat inner-product index.
5. **Retrieve**: Query the index and fetch top-k chunks.
6. **Generate**: Answer with and without retrieved context.
7. **Failure mode**: Show hallucination vs grounded response.

### Environment Setup

We verify the GPU and install the libraries needed for embedding,
indexing, and generation.

In [ ]:
#@title Check GPU and install packages
!nvidia-smi
!pip -q install sentence-transformers faiss-gpu
!pip -q install transformers torch

In [ ]:
#@title Imports
import time
import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")

### Synthetic Knowledge Corpus

We define 20 short documents covering GPU computing, machine learning,
quantitative finance, and Python tooling. Every document is generated
inside the notebook — no external files or downloads required.

In [ ]:
#@title Define documents
docs = [
    "Google Colab provides hosted Jupyter notebooks with optional "
    "GPU acceleration. It is widely used for machine learning, deep "
    "learning, data science, and teaching.",
    "GPUs accelerate workloads with high degrees of parallelism, "
    "such as neural network training, image generation, Monte Carlo "
    "simulation, and large matrix operations.",
    "RAPIDS is a suite of GPU-accelerated data-science libraries built "
    "on CUDA. cuDF provides a pandas-like API that runs on the GPU.",
    "PyTorch is an open-source deep-learning framework that supports "
    "dynamic computation graphs, making it popular for research and "
    "production.",
    "The Black-Scholes-Merton model prices European options under the "
    "assumption that the underlying asset follows geometric Brownian "
    "motion with constant volatility and risk-free rate.",
    "Monte Carlo simulation estimates option prices by generating "
    "many random price paths and averaging discounted payoffs. GPUs "
    "make this embarrassingly parallel.",
    "A barrier option is a path-dependent exotic option whose payoff "
    "depends on whether the underlying asset breaches a specified "
    "price level during the life of the option.",
    "Greeks measure the sensitivity of an option price to changes in "
    "market parameters. Delta is the first derivative with respect "
    "to the underlying spot price.",
    "Large language models (LLMs) are neural networks with billions "
    "of parameters trained on vast text corpora. They can generate "
    "text, summarise documents, and answer questions.",
    "LoRA (Low-Rank Adaptation) freezes the weights of a pre-trained "
    "model and injects small trainable rank-decomposition matrices "
    "into each layer, dramatically reducing memory and compute.",
    "Retrieval-augmented generation (RAG) enhances LLM output by "
    "retrieving relevant documents from a knowledge base before "
    "generation, reducing hallucination and improving factual accuracy.",
    "FAISS is a library for efficient similarity search and "
    "clustering of dense vectors. It supports GPU indices and can "
    "search millions of embeddings in milliseconds.",
    "Sentence-transformers encode text into dense vectors such that "
    "semantically similar sentences are close in vector space. The "
    "all-MiniLM-L6-v2 model is a fast 22-million-parameter encoder.",
    "Stable Diffusion is a latent diffusion model that generates "
    "images from text prompts. It uses a variational autoencoder "
    "to compress images into a latent space.",
    "A convolutional neural network (CNN) applies learned filters to "
    "local regions of an image, automatically extracting features "
    "such as edges, textures, and shapes.",
    "Fashion-MNIST is a drop-in replacement for MNIST containing "
    "28x28 grayscale images of clothing items. It is commonly used "
    "to benchmark image-classification models.",
    "The Transformer architecture replaced recurrent layers with "
    "self-attention mechanisms, enabling parallel training and "
    "state-of-the-art performance on NLP and vision tasks.",
    "Time-series forecasting predicts future values from historical "
    "observations. Models include ARIMA, LSTM, Transformer, and "
    "temporal convolutional networks.",
    "yfinance is a Python library that downloads historical market "
    "data from Yahoo Finance. It is widely used for quantitative "
    "finance research and backtesting.",
    "The AI Engineer is a training programme by Dr. Yves J. Hilpisch "
    "that covers AI, machine learning, deep learning, and "
    "generative AI for engineers and quants.",
]
print(f"Corpus: {len(docs)} documents")

### Chunking and Embedding

Each document becomes one chunk. We encode them with
`sentence-transformers/all-MiniLM-L6-v2` into 384-dimensional vectors.

In [ ]:
#@title Encode documents
encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device,
)

embeddings = encoder.encode(
    docs,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f"Embeddings shape: {embeddings.shape}")

### Build the FAISS Index

We use a flat inner-product index. Because embeddings are L2-normalised,
inner product equals cosine similarity.

In [ ]:
#@title Create FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

# Use GPU index if available
if device == "cuda":
    res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)

index.add(np.asarray(embeddings, dtype="float32"))
print(f"Index size: {index.ntotal} vectors, dim={dim}")

### Retrieval Function

Given a user question, encode it and search the index for the top-k
most similar chunks.

In [ ]:
#@title Retrieve relevant chunks
def retrieve(query, k=3):
    q_emb = encoder.encode(
        [query],
        normalize_embeddings=True,
    )
    q_emb = np.asarray(q_emb, dtype="float32")
    scores, idxs = index.search(q_emb, k)
    return [docs[i] for i in idxs[0]], scores[0]


query = "What is a barrier option?"
chunks, scores = retrieve(query, k=3)
print(f"Query: {query}\n")
for i, (chunk, score) in enumerate(zip(chunks, scores), 1):
    print(f"--- Result {i} (score={score:.3f}) ---")
    print(chunk[:200])
    print()

### Load a Small LLM for Generation

We use `Qwen2.5-1.5B-Instruct` (~3 GB in `bfloat16`) — the same lightweight
model from `01_local_llm_colab_gpu.ipynb`.

In [ ]:
#@title Load model and tokenizer
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True
)

dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True,
).to(device) if device == "cpu" else AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

print(f"LLM loaded on {device}")

### Generation with and without Retrieval

We build two prompts: one with only the question (baseline), and one
with the top-3 retrieved chunks injected as context.

In [ ]:
#@title Generation helper
def generate(prompt, max_new=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(
        llm.device
    )
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(
        out[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    return text.strip()


def build_rag_prompt(question, chunks):
    context = "\n\n".join(
        f"{i + 1}. {c}"
        for i, c in enumerate(chunks)
    )
    return (
        f"Use the following context to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )


QUESTION = "What is a barrier option and how does it differ "
QUESTION += "from a European option?"

chunks, _ = retrieve(QUESTION, k=3)
rag_prompt = build_rag_prompt(QUESTION, chunks)
base_prompt = f"{QUESTION}\nAnswer:"

In [ ]:
#@title Baseline (no retrieval)
print("=" * 60)
print("BASELINE: NO RETRIEVAL")
print("=" * 60)
base_answer = generate(base_prompt)
print(base_answer)

In [ ]:
#@title RAG (with retrieval)
print("\n" + "=" * 60)
print("RAG: WITH RETRIEVED CONTEXT")
print("=" * 60)
rag_answer = generate(rag_prompt)
print(rag_answer)

### Exercises

1. **More documents**: Add 10 more synthetic documents about a topic
   you care about. Does retrieval quality improve?
2. **Larger k**: Try `k=5` or `k=10`. Does the answer become more
   detailed or more noisy?
3. **Re-ranking**: After retrieving `k=10` chunks, keep only the top-3
   by a second scoring pass (e.g. keyword overlap with the question).
4. **Different encoder**: Swap `all-MiniLM-L6-v2` for
   `BAAI/bge-small-en-v1.5`. Do the retrieved chunks change?
5. **Chunking strategy**: Split long documents into smaller sentences
   and index each sentence separately. How does this affect precision?

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
